In [443]:
import random
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, clear_output
import ipywidgets as widgets
from IPython.display import Audio 
import os #song pull uses os to check if a song file is in the music folder

In [444]:
badges = pd.read_csv('data/badges.csv')
badges

,badge,power,kind,min_value,max_value,min_value2,max_value2,formula,image,what_it_does
0,Fire Safety,Fireball,attack,1.00,3.0,22.0,25.0,small = random min_value to max_value (1 to 3)...,badge_fire_safety.png,Wild flames! The fire rolls a small number (1 ...
1,Electricity,Lightning Hands,attack,8.00,14.0,NaN,NaN,their health - random 8 to 14,badge_electricity.png,Shoot lightning bolts at the enemy. Your attac...
2,Astronomy,Meteor Strike,attack,5.00,18.0,NaN,NaN,their health - random 5 to 18,badge_astronomy.png,Call down meteors from the sky. Your attack ta...
3,Chemistry,Potion Bombs,attack,10.00,10.0,NaN,NaN,their health - 10,badge_chemistry.png,Throw exploding potion flasks. Your attack ALW...
4,Archery,Rain of Arrows,attack,0.00,36.0,NaN,NaN,their health - random 0 to 36,badge_archery.png,Fire a rain of glowing arrows. Your attack tak...
5,First Aid,Healing Aura,health,0.00,50.0,NaN,NaN,your health + random min_value to max_value (0...,badge_first_aid.png,A green glow heals you a random amount between...
6,Lifesaving,Revive,health,5.00,18.0,NaN,NaN,knocked out this round: your health = max_valu...,badge_lifesaving.png,Switch it on for one round. If you get knocked...
7,Camping,Base Camp,health,1.00,1.0,NaN,NaN,your health increase = their attack x max_valu...,badge_camping.png,Hide in your tent this round. Whatever attack ...
8,Dog Care,Guard Dog,special,0.00,0.0,NaN,NaN,their attack x max_value (0),badge_dog_care.png,"A cute dog distracts the other player, so thei..."
9,Rover Repair,Rovey Boost,special,2.00,2.0,NaN,NaN,your attack x max_value (2),badge_rover_repair.png,Rovey rolls in next to you and DOUBLES your at...


In [445]:
#two screen boxes 
screen = widgets.Output()
talk = widgets.Output()

In [446]:
#rovey talking
def rovey(text):
    display(widgets.HTML('<p style="font-size:18px; line-height:1.6">Rovey: ' + text + '</p>'))

In [447]:
#what loads pictures 
def picture(file, width):
    data = open('images/' + file, 'rb').read()
    return widgets.Image(value=data, width=width)

In [448]:
#shows pics
def show(file, width):
    display(picture(file, width))

In [449]:
#finds badges
def get_badge(name):
    return badges[badges['badge'] == name].iloc[0]

In [450]:
#lists badge per kind 
def names_of(kind):
    return list(badges[badges['kind'] == kind]['badge'])

In [451]:
#button for boy scout evil shows up fully wihtout this it cuts out 
def make_button(text, action):
    b = widgets.Button(description=text, layout=widgets.Layout(width='auto'))
    b.on_click(action)
    return b

In [452]:
#small button under each badge that epxlains what it does 

answer_boxes = {}

def explain(b):
    row = get_badge(b.tooltip)
    box = answer_boxes[b.tooltip]
    with box:
        clear_output()
        display(widgets.HTML('<b style="color:red">What did I just click?</b>'))
        rovey(row['badge'] + ' (' + row['power'] + '): ' + row['what_it_does'])

In [453]:
#makes it a question button 
def question_button(name):
    q = widgets.Button(description='?', tooltip=name)
    q.on_click(explain)
    box = widgets.Output(layout=widgets.Layout(width='200px'))
    answer_boxes[name] = box
    return widgets.VBox([q, box])

In [454]:
#makes health bar also ensures that color changes as it goes down 
def health_bar(player):
    hp = player['hp']
    if hp > 60:
        color = 'success'
    elif hp > 30:
        color = 'warning'
    else:
        color = 'danger'
    bar = widgets.IntProgress(value=hp, max=100, bar_style=color, description=player['name'], style={'description_width': '100px'})
    display(widgets.HBox([bar, widgets.Label(str(hp) + ' / 100')]))

In [455]:
#function that updates health bar 
def change_hp(player, amount):
    player['hp'] = player['hp'] + amount
    if player['hp'] > 100:
        player['hp'] = 100
    if player['hp'] < 0:
        player['hp'] = 0

In [456]:
#establish players 
girl = {}
boy = {}

In [457]:
#set up for main player  and opponent bot
def new_game():
    #name stays as what you typed when you play again and is Girl Scout only if no name was typed yet
    if 'name' not in girl:
        girl['name'] = 'Girl Scout'
    girl['pic'] = 'girl'
    girl['hp'] = 100
    girl['outfit'] = ''
    girl['attacks'] = []
    girl['health'] = ''
    girl['special'] = ''
    girl['special2'] = ''
    girl['used_health'] = False
    girl['used_special'] = False
    girl['used_special2'] = False
    girl['revive'] = False
    girl['extra_use'] = ''

    boy['name'] = 'Boy Scout'
    boy['pic'] = 'boy'
    boy['hp'] = 100
    boy['outfit'] = 'boy_scout.png'
    boy['attacks'] = random.sample(names_of('attack'), 2)
    boy['health'] = random.choice(names_of('health'))
    specials = random.sample(names_of('special'), 2)
    boy['special'] = specials[0]
    boy['special2'] = specials[1]
    boy['used_health'] = False
    boy['used_special'] = False
    boy['used_special2'] = False
    boy['revive'] = False
    boy['extra_use'] = ''
    boy['rage'] = False
    boy['rage_shown'] = False

In [458]:
#song pull log loads every pre made song from the music folder into songs so the game can play them
songs = {}
song_list = ['picking', 'loadout', 'battle1', 'battle2', 'battle3', 'battle4', 'battle5', 'winner', 'loser', 'salsa', 'rage', 'intro', 'stolen', 'thief']
for key in song_list:
    #song pull if the song file is in the music folder it gets loaded and if it is missing the game just skips it
    if os.path.exists('music/' + key + '.mp3'):
        songs[key] = open('music/' + key + '.mp3', 'rb').read()

In [459]:
#box where songs will play 
music = widgets.Output()

def play_music(key):
    if songs.get(key) != None:
        with music:
            clear_output()
            #song pull plays the song straight from the file that was loaded from the music folder
            display(Audio(songs[key], autoplay=True))

#song pull plays the start menu song on repeat until the cookie stealing scene
def play_intro_music():
    if songs.get('intro') != None:
        with music:
            clear_output()
            display(widgets.Audio(value=songs['intro'], format='mp3', autoplay=True, loop=True))

def play_battle_music():
    ready = []
    for key in ['battle1', 'battle2', 'battle3', 'battle4', 'battle5']:
        if songs.get(key) != None:
            ready.append(key)
    if len(ready) > 0:
        play_music(random.choice(ready))

In [460]:
#assks for name so it can be used elsewhere 
def show_name(b):
    with screen:
        clear_output()
        show('rovey_hello.png', 400)
        rovey('Before we start... what is your name?')
        name_box = widgets.Text(placeholder='Type your name')

        def lock_name(b):
            name = name_box.value.strip()
            if name == '':
                rovey('Type your name first!')
                return
            #name you type becomes the girl scout name in the game
            girl['name'] = name
            show_hello(b)

        display(name_box)
        display(make_button("Let's go!", lock_name))

In [461]:
#create menu
def show_menu(b=None):
    new_game()
    with screen:
        clear_output()
        #song pull starts the start menu song when the game loads
        play_intro_music()
        show('menu_mars.png', 600)
        display(make_button('Ready to begin?', show_name)) 

In [462]:
#rovey first questions 
def show_hello(b):
    with screen:
        clear_output()
        show('rovey_hello.png', 400)
        rovey('Hello, I am Rovey, your Martian rover companion.')
        rovey('I have a very important question...')
        rovey('Did you steal the cookie from the cookie jar?')
        display(make_button('Yes', show_cookie_thief))
        display(make_button('Who me?', show_who_me))

In [463]:
def show_who_me(b):
    with screen:
        clear_output()
        show('rovey_hello.png', 400)
        rovey('Yes, you.')
        display(make_button('Yes', show_cookie_thief))
        
        display(make_button("Couldn't be", show_then_who))

In [464]:
#if they say they did steal the cookie form cookie jar? 
def show_cookie_thief(b):
    with screen:
        clear_output()
        #song pull stops the start menu song and plays the cookie thief song when you say you stole the cookie
        play_music('thief')
        show('rovey_how_could_you.png', 400)
        rovey("I'm sad... how could you?")
        print('GAME OVER')
        display(make_button('Play again', show_menu))

In [465]:
def show_then_who(b):
    with screen:
        clear_output()
        show('rovey_questioning.png', 400)
        rovey('Then who...')
        display(make_button('OK, it was me', show_cookie_thief))
        display(make_button('It was the evil Boy Scouts', show_mission))

In [466]:
#roey talks about th emission
def show_mission(b):
    with screen:
        clear_output()
        #song pull stops the start menu song and plays the cookie stolen song
        play_music('stolen')
        show('boy_steals_cookies.png', 400)
        rovey('We need to defeat them and get back our cookie glory.')
        rovey('Are you ready?')
        display(make_button('Yes!', show_outfits))

In [467]:
def show_outfits(b):
    with screen:
        clear_output()
        show('rovey_happy.png', 200)
        rovey('OK, pick your outfit!')
        box_a = widgets.VBox([picture('girl_outfit_a.png', 200), make_button('Outfit A', choose_outfit)])
        box_b = widgets.VBox([picture('girl_outfit_b.png', 200), make_button('Outfit B', choose_outfit)])
        box_c = widgets.VBox([picture('girl_outfit_c.png', 200), make_button('Outfit C', choose_outfit)])
        display(widgets.HBox([box_a, box_b, box_c]))

In [468]:
#pic outfit (but i got too ambitiouse about the image generations it owudl require oto many 
def choose_outfit(b):
    if b.description == 'Outfit A':
        girl['outfit'] = 'girl_outfit_a.png'
    if b.description == 'Outfit B':
        girl['outfit'] = 'girl_outfit_b.png'
    if b.description == 'Outfit C':
        girl['outfit'] = 'girl_outfit_c.png'
    pick_badges('attack', 2)

In [469]:
#picks badges and (powwers) and health and special 
def pick_badges(kind, how_many):
    with screen:
        clear_output()
        if kind == 'attack': #checks if you are in first badge scene 
            play_music('picking') #plays it
        show('rovey_happy.png', 200)
        rovey('Pick ' + str(how_many) + ' ' + kind + ' badge. Click ? to see what it does.')

        toggles = []
        cards = []
        for name in names_of(kind):
            pick = widgets.ToggleButton(description=name)
            toggles.append(pick)
            cards.append(widgets.VBox([picture(get_badge(name)['image'], 150), pick, question_button(name)]))

        def on_next(b):
            picked = []
            for t in toggles:
                if t.value == True:
                    picked.append(t.description)

            if len(picked) != how_many:
                with talk:
                    clear_output()
                    if len(picked) == 0:
                        rovey('No! You need to pick ' + str(how_many) + '.')
                    else:
                        rovey('Pick ' + str(how_many) + '! You picked ' + str(len(picked)) + '.')
                return

            if kind == 'attack':
                girl['attacks'] = picked
                pick_badges('health', 1)
            elif kind == 'health':
                girl['health'] = picked[0]
                pick_badges('special', 2)
            else:
                girl['special'] = picked[0]
                girl['special2'] = picked[1]
                show_loadout()

        display(widgets.HBox(cards))
        display(make_button('Next', on_next))
        talk.clear_output()
        display(talk)
    

In [470]:
#places badge in the righ tplace
def paste(scene, file, x, y, size):
    pic = Image.open('images/' + file).convert('RGBA')
    pic.thumbnail((size, size))
    scene.paste(pic, (x, y), pic)

In [471]:
#create and show th eloadout! also labels position if its atatck health or special 
def make_loadout(player):
    scene = Image.open('images/bg_battle.png').convert('RGBA').resize((900, 600))
    paste(scene, player['outfit'], 30, 60, 520)

    draw = ImageDraw.Draw(scene)
    font = ImageFont.load_default(size=26)
    draw.text((715, 10), 'ATTACKS', font=font, fill='white', stroke_width=3, stroke_fill='black', anchor='mt')
    draw.text((715, 200), 'HEALTH ABILITY', font=font, fill='white', stroke_width=3, stroke_fill='black', anchor='mt')
    draw.text((715, 390), 'SPECIAL ABILITIES', font=font, fill='white', stroke_width=3, stroke_fill='black', anchor='mt')

    paste(scene, get_badge(player['attacks'][0])['image'], 570, 45, 140)
    paste(scene, get_badge(player['attacks'][1])['image'], 720, 45, 140)
    paste(scene, get_badge(player['health'])['image'], 645, 235, 140)
    paste(scene, get_badge(player['special'])['image'], 570, 425, 140)
    paste(scene, get_badge(player['special2'])['image'], 720, 425, 140)
    return scene

In [472]:
def status(name, used):
    if used == True:
        return '<s>' + name + '</s> (used up)'
    return name

In [473]:
#this allows you to see what your oppenonet chose and has left on the right top side 
def show_inventory(player):
    display(widgets.HTML('<div style="background:#fff3cd; border:3px solid orange; border-radius:10px; padding:10px 16px; font-size:16px; line-height:1.6"><b style="color:#c0392b; font-size:18px">' + player['name'] + ' has:</b><br>'
        + 'Attacks: ' + player['attacks'][0] + ', ' + player['attacks'][1] + '<br>'
        + 'Health ability: ' + status(player['health'], player['used_health']) + '<br>'
        + 'Special abilities: ' + status(player['special'], player['used_special']) + ', ' + status(player['special2'], player['used_special2']) + '</div>'))

In [474]:
#tells what each has and explains ?
def tell_picks(player, who):
    rovey(who + ' picked:')
    for name in player['attacks'] + [player['health'], player['special'], player['special2']]:
        row = get_badge(name)
        text = widgets.HTML('<b>' + row['kind'].upper() + ': ' + name + ' - ' + row['power'] + '</b><br>' + row['what_it_does'], layout=widgets.Layout(width='700px'))
        display(widgets.HBox([picture(row['image'], 60), text], layout=widgets.Layout(align_items='center')))

In [475]:
#makes the tell picsk look nicer and bigger 
def big(text):
    display(widgets.HTML('<h2>Rovey: ' + text + '</h2>'))

In [476]:
#this hsows th eloadout
def show_loadout():
    with screen:
        clear_output()
        play_music('loadout') #plays loadout song
        big('Look at you! Ready to fight.')
        display(make_loadout(girl))
        tell_picks(girl, 'You')
        big('And here is the evil Boy Scout you will fight:')
        display(make_loadout(boy))
        tell_picks(boy, 'The Boy Scout')
        big("Good luck. Don't let us down!")
        big('Every Girl Scout is counting on you to get the cookies back.')
        display(make_button('Start the battle!', show_round))

In [477]:
#when abdges show in befor eatatck if you forgot what each does you can click
def badge_cards(names, toggles):
    cards = []
    for name in names:
        t = widgets.ToggleButton(description=name)
        toggles.append(t)
        cards.append(widgets.VBox([picture(get_badge(name)['image'], 120), t, question_button(name)], layout=widgets.Layout(align_items='center')))
    return widgets.HBox(cards)

In [478]:
def picked_names(toggles):
    picked = []
    for t in toggles:
        if t.value == True:
            picked.append(t.description)
    return picked

In [479]:
#roudn explaantion betterment or make it look nice 
def show_round(b):
    with screen:
        clear_output()
        play_battle_music() #plays song
        left = widgets.Output()
        right = widgets.Output()
        with left:
            health_bar(girl)
            health_bar(boy)
        with right:
            show_inventory(boy)
        display(widgets.HBox([left, right], layout=widgets.Layout(width='100%', justify_content='space-between')))

        rovey('Every round you MUST pick 1 attack.')
        rovey('You can ALSO use your health badge OR your special badge. Not both.')
        rovey('Each of those works only once per game.')

        extras = []
        if girl['used_health'] == False:
            extras.append(girl['health'])
        if girl['used_special'] == False:
            extras.append(girl['special'])
        if girl['used_special2'] == False:
            extras.append(girl['special2'])

        attack_toggles = []
        extra_toggles = []
        talk.clear_output()
        display(talk)
        rovey('Pick 1 attack:')
        display(badge_cards(girl['attacks'], attack_toggles))
        rovey('Pick 1 extra, or leave it empty for nothing:')
        display(badge_cards(extras, extra_toggles))

        def lock_in(b):
            attacks = picked_names(attack_toggles)
            picked_extras = picked_names(extra_toggles)

            if len(attacks) != 1:
                with talk:
                    clear_output()
                    rovey('Pick exactly 1 attack!')
                return
            if len(picked_extras) > 1:
                with talk:
                    clear_output()
                    rovey('Only 1 extra! Health OR special, not both.')
                return

            extra = 'Nothing'
            if len(picked_extras) == 1:
                extra = picked_extras[0]
            play_round(attacks[0], extra)

        display(make_button('Lock it in!', lock_in))
  

In [493]:
#shows player health or abilities used 
def used_key(player, name):
    if name == player['health']:
        return 'used_health'
    if name == player['special']:
        return 'used_special'
    return 'used_special2'

In [481]:
#the logic if you acitvate magic badge or magic trick badge 
def use_extra(player, extra):
    if extra == 'Nothing':
        return
    if extra == player['extra_use']:
        player['extra_use'] = ''
        print('Magic! ' + extra + ' was doubled, so this was the extra use.')
    else:
        player[used_key(player, extra)] = True
    print(player['name'] + ' activates ' + extra + '!')

    if extra == 'First Aid':
        heal = random.randint(int(get_badge(extra)['min_value']), int(get_badge(extra)['max_value']))
        change_hp(player, heal)
        show(player['pic'] + '_first_aid.png', 300)
        print(player['name'] + ' heals +' + str(heal))
        health_bar(player)

    if extra == 'Magician':
        #logic to ensure it doubles a randomly selected ability form their load out
        others = [player['health'], player['special'], player['special2']]
        others.remove('Magician')
        pick = random.choice(others)
        show(player['pic'] + '_magician.png', 300)
        if player[used_key(player, pick)] == True:
            player[used_key(player, pick)] = False
        else:
            player['extra_use'] = pick
        print('Abracadabra! Magic doubles ' + pick + ', so it can be used one extra time starting next round.')

    if extra == 'Lifesaving':
        player['revive'] = True
        show(player['pic'] + '_lifesaving.png', 300)
        comeback = int(get_badge(extra)['max_value'])
        bonus = int(get_badge(extra)['min_value'])
        print('If ' + player['name'] + ' gets knocked out this round, they come back with ' + str(comeback) + ' health. If not, they get +' + str(bonus) + '.')

In [482]:
#math set up for each attack one player at time first girl scout second boy scout 
#correction made to ensure dog show sup and doesnt add the attack 
def do_attack(attacker, defender, attack_name, attacker_extra, defender_extra):
    row = get_badge(attack_name)
    if pd.notna(row['min_value2']):
        small = random.randint(int(row['min_value']), int(row['max_value']))
        big = random.randint(int(row['min_value2']), int(row['max_value2']))
        damage = random.choice([small, big])
    else:
        damage = random.randint(int(row['min_value']), int(row['max_value']))

    #when dog is activated no attack is shown because they forgot to attack
    if defender_extra == 'Dog Care':
        damage = damage * int(get_badge('Dog Care')['max_value'])
        show('dog_pet_by_' + attacker['pic'] + '.png', 400)
        print(defender['name'] + ' used Dog Care! ' + attacker['name'] + ' got distracted by the puppy and forgot to attack! Attack = ' + str(damage))
        change_hp(defender, -damage)
        health_bar(girl)
        health_bar(boy)
        return

    if pd.notna(row['min_value2']):
        print('The attack rolled ' + str(small) + ' or ' + str(big) + '... and picked ' + str(damage) + '!')
    attack_key = attack_name.lower().replace(' ', '_')
    print(attacker['name'] + ' uses ' + row['power'] + '!')

    if attacker_extra == 'Rover Repair':
        times = int(get_badge('Rover Repair')['max_value'])
        damage = damage * times
        print('Rover Repair multiplies the attack by ' + str(times) + '!')

    #rage mode logi doubles attack
    if attacker == boy and boy['rage'] == True:
        times = int(get_badge('Rage Mode')['max_value'])
        damage = damage * times
        print('RAGE MODE! The Boy Scout hits ' + str(times) + 'x as hard!')

    if defender_extra == 'Salsa Dance':
        result = 'salsa'
    elif defender_extra == 'Camping':
        result = 'camp'
    elif attacker_extra == 'Rover Repair':
        result = 'rover'
    else:
        result = 'hit'

    show(attacker['pic'] + '_' + attack_key + '_' + result + '.png', 400)

    if result == 'salsa':
        if defender == girl: #this ensures that the generate salsa song easter egg is only played if the girl scout activates it not the boyscout
            play_music('salsa') #plays easter egg
        bounce = damage * int(get_badge('Salsa Dance')['max_value'])
        change_hp(attacker, -bounce)
        print('Salsa Dance bounces it back! ' + attacker['name'] + ' loses ' + str(bounce) + '!')
    elif result == 'camp':
        gain = damage * int(get_badge('Camping')['max_value'])
        change_hp(defender, gain)
        print(defender['name'] + ' hides in Base Camp. The attack turns into +' + str(gain) + ' health!')
    else:
        change_hp(defender, -damage)
        print(defender['name'] + ' loses ' + str(damage) + ' health!')

    health_bar(girl)
    health_bar(boy)

In [483]:
#life savior if they did ghet knoeded out so they come back 
def knocked_out(player):
    if player['hp'] > 0:
        return False
    show(player['pic'] + '_passed_out.png', 300)
    print(player['name'] + ' got knocked out!')
    if player['revive'] == True:
        player['revive'] = False
        player['hp'] = int(get_badge('Lifesaving')['max_value'])
        show(player['pic'] + '_revive.png', 300)
        print('...but Lifesaving brings them back with ' + str(player['hp']) + ' health!')
        return False
    return True

In [484]:
#what happens when you activate lifesavor but didn tget knoecket out
def lifesaving_bonus(player):
    if player['revive'] == True:
        player['revive'] = False
        bonus = int(get_badge('Lifesaving')['min_value'])
        change_hp(player, bonus)
        print(player['name'] + ' was not knocked out, so Lifesaving gives +' + str(bonus) + ' health!')
        health_bar(player)

In [485]:
#if there is a tie this image shows up and this logic is put in place to flip a coin 
def show_tie():
    show('tie_coin_flip.png', 400)
    rovey('This will be settled by flipping a coin!')
    rovey('Pick heads or tails, then hit Flip!')
    pick = widgets.ToggleButtons(options=['Heads', 'Tails'])

    def flip(b):
        coin = random.choice(['Heads', 'Tails'])
        with screen:
            clear_output()
            rovey('The coin landed on ' + coin + '!')
            if pick.value == coin:
                show_end(girl)
            else:
                show_end(boy)

    display(pick)
    display(make_button('Flip!', flip))

In [486]:
#sets up the loosing istuation
def game_is_over():
    boy_out = knocked_out(boy)
    girl_out = knocked_out(girl)
    if boy_out and girl_out:
        show_tie()
        return True
    if boy_out:
        show_end(girl)
        return True
    if girl_out:
        show_end(boy)
        return True
    return False

In [487]:
#plays out the round and pics the boys attack
def play_round(girl_attack, girl_extra):
    boy_attack = random.choice(boy['attacks'])
    boy_extras = ['Nothing']
    if boy['used_health'] == False:
        boy_extras.append(boy['health'])
    if boy['used_special'] == False:
        boy_extras.append(boy['special'])
    if boy['used_special2'] == False:
        boy_extras.append(boy['special2'])
    boy_extra = random.choice(boy_extras)

    with screen:
        clear_output()
        use_extra(girl, girl_extra)
        use_extra(boy, boy_extra)

        do_attack(girl, boy, girl_attack, girl_extra, boy_extra)
        if game_is_over():
            return

        #ensures rage mode acitvates once they get hit and before they hit back
        #rage mode activting trigger
        rage = get_badge('Rage Mode')
        rage2 = get_badge('Rage Mode 2')
        rage3 = get_badge('Rage Mode 3')
        rage4 = get_badge('Rage Mode 4')
        #different rage mode triggers
        if boy['hp'] < int(rage['min_value']) and girl['hp'] > int(rage['min_value2']):
            boy['rage'] = True
        if boy['hp'] < int(rage2['min_value']) and girl['hp'] > int(rage2['min_value2']):
            boy['rage'] = True
        if girl['hp'] > boy['hp'] * float(rage3['min_value']):
            boy['rage'] = True
        #rage mode trigger 4 when the girl scout is winning by more than 36 from a big attack or from healing
        if girl['hp'] - boy['hp'] > int(rage4['min_value']):
            boy['rage'] = True

        #rage mode scene shows 
        if boy['rage'] == True:
            #once rage mode shows in the story line it wont show every round
            if boy['rage_shown'] == False:
                boy['rage_shown'] = True
                print('RAGE MODE!')
                rovey("The Boy Scout knows he's losing... but look!! He is cheating, he is drinking a rage potion!")
                show('boy_rage_potion.png', 300)
                rovey('Oh nooo, he is in RAGE MODE! His attacks hit ' + str(int(rage['max_value'])) + 'x as hard for the rest of the game! Focus ' + girl['name'] + ', you got this, lock in!')
                show('boy_rage.png', 300)
                #esnures music only plays one in rage acitvation regardless if rage stays activated in further rounds
                #song pull when rage mode turns on the rage song plays
                play_music('rage')

        do_attack(boy, girl, boy_attack, boy_extra, girl_extra)
        if game_is_over():
            return

        lifesaving_bonus(girl)
        lifesaving_bonus(boy)
        display(make_button('Next round', show_round))

In [488]:
#if you loose and want to play again this restaurst without havign to go though tht ehwle cookie ajr intro again 
def play_again(b):
    new_game()
    show_outfits(b)
        #if music still plays at end and person is inpatient to list to the mazing creation of victory or looser song it stops it to start again
    music.clear_output()

In [489]:
#end game over mazel tov or you disapoint
def show_end(winner):
    print('GAME OVER')
    if winner == girl:
        play_music('winner') #winner winner chicken dinner song 
        rovey('MAZEL TOV! You got the cookies back!')
        show('girl_winner.png', 400)
    else:
        play_music('loser') #take the l song 
        rovey('You are a disappointment.')
        show('boy_winner.png', 400)
    display(make_button('Play again', play_again))

In [490]:
display(music) #plays song when there is song to play
display(screen)
show_menu()

Output()

Output()